# Credit Default Risk Classification with Support Vector Machines

This notebook builds an end-to-end **Support Vector Classification (SVC)** workflow for estimating credit default risk. The analysis combines exploratory inspection, missing-value handling, feature preprocessing, model selection through cross-validation, and classification-based evaluation.

### Project objective

The goal is to learn a decision boundary that separates customers according to the target variable **default_risk**. Because the dataset contains both numerical and categorical predictors, the workflow uses separate preprocessing strategies before fitting the classifier.

---

## 1. Environment Setup

Import the core libraries used for tabular data manipulation, numerical operations, and visualization. Inline plotting keeps generated charts directly inside the notebook.

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline

## 2. Load the Credit Risk Dataset

Read the source CSV file into a pandas DataFrame. Each row represents a customer record, while the columns contain customer attributes and the credit-risk outcome used for supervised learning.

In [2]:
df= pd.read_csv("svm_zorlayici_kredi_riski.csv")

## 3. Initial Data Exploration

Before modeling, inspect the dataset from several complementary perspectives:

- Preview the first observations to understand the column structure and typical values.
- Review data types and non-null counts to identify numerical, categorical, and incomplete fields.
- Count missing values column by column.
- Examine descriptive statistics to understand scale, spread, and potential extreme values.

These checks guide the preprocessing decisions used later in the pipeline.

In [3]:
df.head()

,customer_id,age,annual_income,debt_to_income,credit_score,employment_years,loan_amount,loan_term_months,late_payments_24m,credit_utilization,region,employment_type,education_level,home_ownership,loan_purpose,device_type,application_hour,default_risk
0,CUST-00001,30,88055.0,0.348,684.0,0.7,38701,36,0.0,0.000,Marmara,Maasli,Yuksek Lisans,NaN,Is Kurma,Android,14,0
1,CUST-00002,18,70131.0,0.400,688.0,1.5,13894,48,1.0,0.154,Dogu Anadolu,Maasli,Lisans,Ev Sahibi,Ihtiyac,Android,5,0
2,CUST-00003,28,49552.0,0.625,NaN,0.0,14345,48,7.0,0.753,Ic Anadolu,Issiz,Onlisans,Ipotekli,Ihtiyac,Android,15,1
3,CUST-00004,50,42547.0,0.193,647.0,20.3,14969,60,0.0,0.282,Marmara,Serbest,Lisans,Kiraci,Tasit,Android,22,0
4,CUST-00005,39,65171.0,0.296,653.0,10.8,73373,60,2.0,0.490,Ege,Serbest,Lisans,Kiraci,Konut,iOS,22,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         2000 non-null   object 
 1   age                 2000 non-null   int64  
 2   annual_income       1942 non-null   float64
 3   debt_to_income      1926 non-null   float64
 4   credit_score        1963 non-null   float64
 5   employment_years    1919 non-null   float64
 6   loan_amount         2000 non-null   int64  
 7   loan_term_months    2000 non-null   int64  
 8   late_payments_24m   1937 non-null   float64
 9   credit_utilization  1934 non-null   float64
 10  region              2000 non-null   object 
 11  employment_type     1972 non-null   object 
 12  education_level     1951 non-null   object 
 13  home_ownership      1967 non-null   object 
 14  loan_purpose        2000 non-null   object 
 15  device_type         1972 non-null   object 
 16  applic

In [5]:
df.isnull().sum()

customer_id            0
age                    0
annual_income         58
debt_to_income        74
credit_score          37
employment_years      81
loan_amount            0
loan_term_months       0
late_payments_24m     63
credit_utilization    66
region                 0
employment_type       28
education_level       49
home_ownership        33
loan_purpose           0
device_type           28
application_hour       0
default_risk           0
dtype: int64

In [6]:
df.describe()

,age,annual_income,debt_to_income,credit_score,employment_years,loan_amount,loan_term_months,late_payments_24m,credit_utilization,application_hour,default_risk
count,2000.000000,1942.000000,1926.000000,1963.000000,1919.00000,2000.000000,2000.00000,1937.000000,1934.000000,2000.000000,2000.000000
mean,39.176500,62714.685891,0.309229,656.102904,9.61037,27559.145000,47.52000,1.821373,0.414330,11.291500,0.153500
std,11.621828,37777.435977,0.145070,38.191529,8.76835,24607.462427,26.35308,2.002864,0.234091,6.933871,0.360559
min,18.000000,2265.000000,0.020000,518.000000,0.00000,5000.000000,12.00000,0.000000,0.000000,0.000000,0.000000
25%,31.000000,42389.500000,0.204000,630.000000,2.10000,13551.000000,36.00000,1.000000,0.259250,5.000000,0.000000
50%,39.000000,56521.000000,0.308000,657.000000,7.60000,21614.500000,36.00000,1.000000,0.405000,11.000000,0.000000
75%,47.000000,75586.000000,0.407000,682.000000,15.00000,33587.750000,60.00000,3.000000,0.548750,17.000000,0.000000
max,75.000000,947540.000000,0.787000,778.000000,45.00000,542395.000000,120.00000,26.000000,2.418000,23.000000,1.000000


### Missing-value inspection

The following expression displays the result of removing incomplete rows without assigning it back to the original DataFrame. The modeling workflow therefore retains the original data and handles missing values later through dedicated imputation steps inside the preprocessing pipelines.

In [7]:
df.dropna()

,customer_id,age,annual_income,debt_to_income,credit_score,employment_years,loan_amount,loan_term_months,late_payments_24m,credit_utilization,region,employment_type,education_level,home_ownership,loan_purpose,device_type,application_hour,default_risk
1,CUST-00002,18,70131.0,0.400,688.0,1.5,13894,48,1.0,0.154,Dogu Anadolu,Maasli,Lisans,Ev Sahibi,Ihtiyac,Android,5,0
3,CUST-00004,50,42547.0,0.193,647.0,20.3,14969,60,0.0,0.282,Marmara,Serbest,Lisans,Kiraci,Tasit,Android,22,0
4,CUST-00005,39,65171.0,0.296,653.0,10.8,73373,60,2.0,0.490,Ege,Serbest,Lisans,Kiraci,Konut,iOS,22,0
5,CUST-00006,20,39107.0,0.510,624.0,1.8,7528,12,4.0,0.863,Karadeniz,Sozlesmeli,Onlisans,Kiraci,Borc Birlestirme,iOS,18,0
6,CUST-00007,45,44900.0,0.068,699.0,4.5,14153,120,2.0,0.517,Karadeniz,Maasli,Lise,Kiraci,Borc Birlestirme,Android,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,CUST-01996,38,63597.0,0.459,678.0,13.8,26004,60,6.0,0.786,Karadeniz,Serbest,Onlisans,Ipotekli,Borc Birlestirme,Android,12,0
1996,CUST-01997,20,35369.0,0.393,696.0,2.6,20066,48,3.0,0.282,Akdeniz,Maasli,Lisans,Ipotekli,Saglik,iOS,10,0
1997,CUST-01998,47,52024.0,0.531,670.0,7.8,30268,60,2.0,0.275,Guneydogu Anadolu,Sozlesmeli,Onlisans,Ev Sahibi,Konut,Android,5,0
1998,CUST-01999,50,84084.0,0.213,679.0,27.6,41678,36,0.0,0.227,Guneydogu Anadolu,Serbest,Lisans,Ev Sahibi,Is Kurma,iOS,6,0


## 4. Modeling Components

Import the scikit-learn tools required for the complete classification workflow:

- **StandardScaler** standardizes numerical features.
- **OneHotEncoder** converts categorical values into machine-readable indicator columns.
- **SimpleImputer** fills missing numerical and categorical values.
- **ColumnTransformer** applies different transformations to different feature groups.
- **Pipeline** keeps preprocessing and modeling steps together, reducing leakage risk.
- **GridSearchCV** compares multiple SVC configurations using cross-validation.
- Classification metrics summarize predictive performance from several perspectives.

In [8]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.pipeline import Pipeline

## 5. Prepare Features and Target

The customer identifier is removed because it acts as a record key rather than a meaningful predictive feature. The remaining columns are then separated into:

- **X** — input features used by the classifier.
- **y** — the **default_risk** target the model is expected to predict.

In [9]:
df.drop("customer_id",axis=1,inplace=True)

In [10]:
X= df.drop("default_risk",axis=1)
y=df["default_risk"]

## 6. Identify Numerical and Categorical Features

Detect column groups directly from their data types. This makes the preprocessing stage systematic and allows each group to receive transformations appropriate to its statistical meaning.

The next two cells display the detected column names as a quick validation step.

In [11]:
numeric_columns = X.select_dtypes(include="number").columns
categorical_columns = X.select_dtypes(include=["string","object","category"]).columns

In [12]:
numeric_columns

Index(['age', 'annual_income', 'debt_to_income', 'credit_score',
       'employment_years', 'loan_amount', 'loan_term_months',
       'late_payments_24m', 'credit_utilization', 'application_hour'],
      dtype='object')

In [13]:
categorical_columns

Index(['region', 'employment_type', 'education_level', 'home_ownership',
       'loan_purpose', 'device_type'],
      dtype='object')

## 7. Build Preprocessing Pipelines

Two specialized pipelines prepare the raw features:

### Numerical features

Numerical values are standardized so that variables measured on different scales contribute more comparably to the SVC decision boundary. Missing numerical values are filled with the median, a robust choice when distributions may contain skew or outliers.

### Categorical features

Categories are one-hot encoded while previously unseen values are ignored safely at prediction time. Missing categorical values are filled using the most frequent category.

Combining these operations in pipelines ensures that the same transformations are learned from training data and consistently applied to test data.

In [14]:
numeric_pipeline = Pipeline([
    ("scaler",StandardScaler()),
    ("impute",SimpleImputer(strategy="median"))
])
categoric_pipeline = Pipeline([
    ("encoder",OneHotEncoder(handle_unknown="ignore")),
    ("impute",SimpleImputer(strategy="most_frequent"))
])

### Combine feature-specific transformations

The column transformer routes numerical and categorical columns through their corresponding pipelines, then concatenates the transformed outputs into one model-ready feature matrix.

In [15]:
preprocessor = ColumnTransformer([
    ("numeric",numeric_pipeline,numeric_columns),
    ("categoric",categoric_pipeline,categorical_columns)
])

## 8. Create Training and Test Sets

Reserve 25% of the observations for final evaluation and use the remaining 75% for training and cross-validation. A fixed random state makes the split reproducible across runs.

Keeping the test set separate provides a more realistic estimate of how the selected model may perform on unseen customer records.

In [16]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=15)

## 9. Configure the Support Vector Classifier

Support Vector Classification searches for a decision boundary that separates classes with the widest possible margin. The standalone estimator display below provides a quick view of its default configuration before hyperparameter tuning.

In [17]:
SVC()

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


### Hyperparameter search space

The grid explores three influential SVC choices:

- **C** controls the trade-off between a wider margin and penalties for classification errors. Smaller values apply stronger regularization; larger values prioritize fitting the training observations.
- **kernel** determines the mathematical form of the decision boundary. Linear and nonlinear alternatives are compared.
- **gamma** controls the influence of individual training samples for kernels that use it.

Testing these combinations allows the model-selection process to adapt its complexity to the dataset.

In [18]:
param_grid = {
    "C"  : [0.01,0.1,1,10,100],
    "kernel": ['rbf','linear', 'poly','sigmoid'],
    "gamma" : ["scale","auto"]
}

## 10. Assemble the End-to-End Model Pipeline

The full pipeline first preprocesses the raw columns and then runs a five-fold grid search over the SVC configurations. Cross-validation repeatedly trains and validates candidate models on different subsets of the training data, while parallel processing speeds up the search when multiple CPU cores are available.

Placing preprocessing inside the pipeline is important: every validation fold learns its transformations only from the corresponding training portion, which helps prevent data leakage.

In [19]:
model_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("grid",GridSearchCV(estimator=SVC(),param_grid=param_grid,cv=5,n_jobs=-1))
])

## 11. Train the Model and Generate Predictions

Fit the complete pipeline on the training set. During this step, preprocessing parameters and the strongest cross-validated SVC configuration are learned together. The fitted pipeline is then used to predict default-risk classes for the held-out test set.

In [20]:
model_pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('grid', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categoric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
y_pred = model_pipeline.predict(X_test)

## 12. Evaluate Classification Performance

Evaluate the predictions with complementary metrics:

- **Accuracy** reports the overall proportion of correct predictions.
- The **classification report** provides precision, recall, and F1-score for each class, helping reveal whether performance is balanced across risk groups.
- The **confusion matrix** shows the counts of correct and incorrect predictions for each actual–predicted class combination.

Accuracy alone can be misleading when one class is more common than the other, so the class-level metrics and confusion matrix should be interpreted alongside it.

### Practical interpretation

- High **precision** for the default-risk class means fewer low-risk customers are incorrectly flagged.
- High **recall** for the default-risk class means fewer genuinely risky customers are missed.
- The preferred balance depends on the business cost of false approvals versus unnecessary rejections.

In [22]:
score = accuracy_score(y_pred,y_test)
print("Accuracy : ",score)
print(classification_report(y_pred,y_test))
print("Confusion Matrix : ",confusion_matrix(y_pred,y_test))

Accuracy :  0.876
              precision    recall  f1-score   support

           0       1.00      0.88      0.93       489
           1       0.14      0.91      0.24        11

    accuracy                           0.88       500
   macro avg       0.57      0.89      0.59       500
weighted avg       0.98      0.88      0.92       500

Confusion Matrix :  [[428  61]
 [  1  10]]
